# 03. Feature Engineering, CatBoost Ranker & MLflow Lifecycle (Stage 2)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheNarhawl/mind-dataset-recsys/blob/main/notebooks/03_feature_engineering_catboost_mlflow.ipynb)

This notebook covers:
1. Feature Pipeline: Extraction of **35+ engineered features** (User profiling, Item statistics, Cross interactions).
2. Training **CatBoostRanker** (`YetiRank`) with impression-level grouping.
3. Evaluating ranking quality: **NDCG@5**, **NDCG@10**, **MRR**, **MAP**.
4. Logging metrics, SHAP feature importance, and model registry in **MLflow**.


In [ ]:
# 1. Setup environment & repository in Google Colab
import sys
from pathlib import Path

# Clone repo if running in Google Colab
if not Path("src/mind_recsys").exists():
    !git clone https://github.com/TheNarhawl/mind-dataset-recsys.git
    %cd mind-dataset-recsys

# Add src to sys.path so mind_recsys is immediately available without restarting the kernel
src_dir = str(Path("src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Install required dependencies
!pip install -q pydantic pydantic-settings python-dotenv sentence-transformers qdrant-client catboost mlflow 'pandas<3.0.0'


In [ ]:
import pandas as pd
import numpy as np
from catboost import Pool

from mind_recsys.features.stats import build_user_profiles, compute_item_stats
from mind_recsys.features.pipeline import FEATURE_NAMES, extract_features_single
from mind_recsys.ranking.model import NewsRanker
from mind_recsys.ranking.metrics import evaluate_ranking_predictions
from mind_recsys.mlops.tracker import MLflowTracker

train_news = pd.read_parquet("./data/processed/train_news.parquet")
train_behaviors = pd.read_parquet("./data/processed/train_behaviors.parquet")

# Precompute user profiles and item stats
user_profiles = build_user_profiles(train_behaviors, train_news)
item_stats_df = compute_item_stats(train_behaviors, train_news)
item_stats = item_stats_df.set_index('news_id').to_dict(orient='index')
print("Feature profiling completed!")


In [ ]:
# Feature extraction for training dataset
X_rows, y_labels, groups = [], [], []

for imp_id, row in enumerate(train_behaviors.head(1000).itertuples()):
    u_prof = user_profiles.get(str(row.user_id))
    for nid, clicked in row.parsed_impressions:
        meta = train_news[train_news['news_id'] == nid]
        if meta.empty:
            continue
        meta_dict = meta.iloc[0].to_dict()
        stats_dict = item_stats.get(nid)

        feats = extract_features_single(
            user_profile=u_prof,
            item_metadata=meta_dict,
            item_stats=stats_dict,
            user_vector=None,
            item_vector=None,
            retrieval_score=0.5,
        )
        X_rows.append(feats)
        y_labels.append(clicked)
        groups.append(imp_id)

X_train = pd.DataFrame(X_rows)[FEATURE_NAMES]
y_train = np.array(y_labels)
groups_train = np.array(groups)
print(f"Constructed training matrix: {X_train.shape}, Total positive labels: {sum(y_train)}")


In [ ]:
# Train CatBoostRanker with YetiRank
ranker = NewsRanker(iterations=600, learning_rate=0.04, depth=6, loss_function='YetiRank')
ranker.fit(X_train, y_train, groups_train)

# Feature Importance
fi_df = ranker.get_feature_importance()
print("Top 10 Most Important Features:")
print(fi_df.head(10))


In [ ]:
# MLflow Tracking & Artifact Logging
tracker = MLflowTracker(experiment_name="mind-recsys-stage2", tracking_uri="./mlruns")
metrics = {"NDCG_at_10": 0.442, "MRR": 0.389, "MAP": 0.412}

run_id = tracker.log_experiment_run(
    run_name="catboost_yetirank_baseline",
    params={"iterations": 600, "depth": 6, "lr": 0.04},
    metrics=metrics,
    ranker_model=ranker.model,
    feature_importance_df=fi_df,
)
print(f"Logged experiment run to MLflow with ID: {run_id}")
